# Pratica — Lezione 5. Predizione senza modello: Monte Carlo e differenze temporali

Questa pratica è tutta tua. Lavorerai in autonomia con la stessa passeggiata casuale che hai costruito in classe e risolverai due esercizi su Monte Carlo (MC) e differenze temporali (TD, dall'inglese Temporal-Difference).

**Come lavorare con questo notebook:**
1. Esegui prima la sezione "Preparazione", che ricostruisce l'ambiente della lezione
2. In ogni esercizio, completa le righe contrassegnate con `# TODO`
3. Esegui la tua soluzione e confronta il risultato con quanto descritto nella consegna
4. Solo dopo, apri la soluzione nascosta per verificare la tua risposta

> **Non guardare la soluzione prima di averci provato.** Il valore di questo esercizio sta nel confrontarsi un po' con la regola di aggiornamento, non nel leggere la risposta.

## Preparazione: recupera la passeggiata casuale della lezione

Queste celle riproducono esattamente l'ambiente che hai costruito nel notebook teorico: la catena di cinque stati interni, i suoi valori reali e `generar_episodio`. Gli esercizi di oggi riutilizzano questi stessi nomi, quindi non cambiarli.

In [ ]:
import numpy as np

N_ESTADOS_INTERNOS = 5
ESTADO_INICIAL = 3          # C, el estado central de la cadena
TERMINAL_IZQUIERDA = 0
TERMINAL_DERECHA = N_ESTADOS_INTERNOS + 1

# Valor real de cada estado interno: V(estado i) = i / (N_ESTADOS_INTERNOS + 1)
VALORES_REALES = np.array(
    [i / (N_ESTADOS_INTERNOS + 1) for i in range(1, N_ESTADOS_INTERNOS + 1)]
)


def generar_episodio():
    '''
    Simula un episodio de paseo aleatorio y devuelve la lista de
    (estado, recompensa) que atraviesa, terminando en el estado terminal.
    '''
    estado = ESTADO_INICIAL
    trayectoria = []
    while True:
        # Con probabilidad 0.5 te mueves a la izquierda o a la derecha
        paso = -1 if np.random.rand() < 0.5 else 1
        siguiente_estado = estado + paso
        # Solo el extremo derecho da recompensa; el resto de pasos dan cero
        recompensa = 1.0 if siguiente_estado == TERMINAL_DERECHA else 0.0
        trayectoria.append((estado, recompensa))
        estado = siguiente_estado
        if estado in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
            break
    return trayectoria


### Controllo rapido

Genera un episodio di esempio per confermare che l'ambiente funziona prima di metterti agli esercizi.

In [ ]:
np.random.seed(0)
print(generar_episodio())


---

## Esercizio 1: completa TD(0) da zero

In classe hai visto la regola di aggiornamento di TD(0):

```
V(S) ← V(S) + α · [R + γ·V(S') − V(S)]
```

In questo esercizio hai già tutta la struttura di `td0_prediccion_ejercicio` scritta —il ciclo sugli episodi, il passo nella passeggiata casuale, il calcolo dell'RMSE—, ma mancano le due righe centrali: calcolare l'errore TD e usarlo per aggiornare V. Completale.

**Suggerimenti:**
- `v_siguiente` è già calcolato per te: vale 0.0 se lo stato successivo è terminale, altrimenti la stima attuale `V[siguiente_estado - 1]`
- L'errore TD è `recompensa + gamma * v_siguiente - V[idx]`
- L'aggiornamento incrementale è `V[idx] += alpha * error_td`
- Se la tua implementazione è corretta, dopo 200 episodi con α = 0.1 l'RMSE finale dovrebbe scendere sotto circa 0.1 (il valore esatto dipende dal seme casuale)

In [ ]:
def td0_prediccion_ejercicio(n_episodios, alpha, gamma=1.0):
    '''Estima V para los estados internos 1..5 con TD(0).'''
    V = np.full(N_ESTADOS_INTERNOS, 0.5)
    historial_rmse = []

    for _ in range(n_episodios):
        estado = ESTADO_INICIAL
        while estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
            paso = -1 if np.random.rand() < 0.5 else 1
            siguiente_estado = estado + paso
            recompensa = 1.0 if siguiente_estado == TERMINAL_DERECHA else 0.0

            if siguiente_estado in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                v_siguiente = 0.0
            else:
                v_siguiente = V[siguiente_estado - 1]

            idx = estado - 1

            # TODO: calcula el error TD: recompensa + gamma * v_siguiente - V[idx]
            error_td = None

            # TODO: actualiza V[idx] sumando alpha * error_td (regla de TD(0))

            estado = siguiente_estado

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


np.random.seed(0)
V_td_ejercicio, historial_td_ejercicio = td0_prediccion_ejercicio(200, alpha=0.1)
print("V estimado:", V_td_ejercicio)
print("Valores reales:", VALORES_REALES)
print(f"RMSE final: {historial_td_ejercicio[-1]:.4f}")


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def td0_prediccion_ejercicio(n_episodios, alpha, gamma=1.0):
    """Stima V per gli stati interni 1..5 con TD(0)."""
    V = np.full(N_ESTADOS_INTERNOS, 0.5)
    historial_rmse = []

    for _ in range(n_episodios):
        estado = ESTADO_INICIAL
        while estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
            paso = -1 if np.random.rand() < 0.5 else 1
            siguiente_estado = estado + paso
            recompensa = 1.0 if siguiente_estado == TERMINAL_DERECHA else 0.0

            if siguiente_estado in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                v_siguiente = 0.0
            else:
                v_siguiente = V[siguiente_estado - 1]

            idx = estado - 1

            # Errore TD: ciò che hai appena osservato (ricompensa + valore
            # scontato dello stato successivo) meno ciò che credevi già
            error_td = recompensa + gamma * v_siguiente - V[idx]

            # Aggiornamento incrementale verso quell'errore, scalato da alpha
            V[idx] += alpha * error_td

            estado = siguiente_estado

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


np.random.seed(0)
V_td_ejercicio, historial_td_ejercicio = td0_prediccion_ejercicio(200, alpha=0.1)
print("V stimato:", V_td_ejercicio)
print("Valori reali:", VALORES_REALES)
print(f"RMSE finale: {historial_td_ejercicio[-1]:.4f}")

# Output atteso approssimativo (con seed 0):
# V stimato: [0.1107 0.3371 0.4691 0.6996 0.8678]
# Valori reali: [0.1667 0.3333 0.5    0.6667 0.8333]
# RMSE finale: 0.0357
```
</details>

---

## Esercizio 2: completa Monte Carlo a prima visita e confronta i due metodi

Ora tocca a MC. La funzione `monte_carlo_primera_visita_ejercicio` percorre ogni episodio dalla fine verso l'inizio per calcolare il ritorno G di ogni passo, e si tiene il ritorno della prima volta in cui compare ciascuno stato. Completa il calcolo del ritorno e l'aggiornamento di V.

Poi, con `td0_prediccion_ejercicio` dell'esercizio precedente già completato, confronterai l'errore finale di MC e di TD(0) con tre diversi tassi di apprendimento: α = 0.01, α = 0.05 e α = 0.15.

**Suggerimenti:**
- Il ritorno si accumula percorrendo la traiettoria all'indietro: `G = recompensa + gamma * G`
- Poiché percorri la traiettoria in ordine inverso, l'ultima assegnazione che fai su `retornos_por_estado[estado]` corrisponde alla prima visita di quello stato nell'episodio: salva lì il G appena calcolato
- L'aggiornamento di V usa la stessa regola incrementale di TD, ma punta al ritorno completo: `V[idx] += alpha * (G - V[idx])`
- Nel confronto finale, aspettati di vedere che un α più alto dà un RMSE più alto con entrambi i metodi: è il rumore residuo di cui avete parlato in classe

In [ ]:
def monte_carlo_primera_visita_ejercicio(n_episodios, alpha, gamma=1.0):
    '''Estima V para los estados internos 1..5 con MC de primera visita.'''
    V = np.full(N_ESTADOS_INTERNOS, 0.5)
    historial_rmse = []

    for _ in range(n_episodios):
        trayectoria = generar_episodio()

        G = 0.0
        retornos_por_estado = {}
        for estado, recompensa in reversed(trayectoria):
            # TODO: acumula el retorno: G = recompensa + gamma * G
            G = None

            # TODO: guarda G como el retorno de "estado" en retornos_por_estado
            #       (recuerda: la última asignación sobre un estado repetido
            #       es la que corresponde a su primera visita)

        for estado, G in retornos_por_estado.items():
            if estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                idx = estado - 1
                # TODO: actualiza V[idx] hacia G con la regla incremental
                #       V[idx] += alpha * (G - V[idx])
                pass

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


np.random.seed(0)
V_mc_ejercicio, historial_mc_ejercicio = monte_carlo_primera_visita_ejercicio(200, alpha=0.1)
print("V estimado:", V_mc_ejercicio)
print("Valores reales:", VALORES_REALES)
print(f"RMSE final: {historial_mc_ejercicio[-1]:.4f}")


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def monte_carlo_primera_visita_ejercicio(n_episodios, alpha, gamma=1.0):
    """Stima V per gli stati interni 1..5 con MC a prima visita."""
    V = np.full(N_ESTADOS_INTERNOS, 0.5)
    historial_rmse = []

    for _ in range(n_episodios):
        trayectoria = generar_episodio()

        G = 0.0
        retornos_por_estado = {}
        for estado, recompensa in reversed(trayectoria):
            # Accumula il ritorno all'indietro dalla fine dell'episodio
            G = recompensa + gamma * G
            # Percorrendo la traiettoria al contrario, l'ultima assegnazione
            # per uno stato ripetuto è quella della sua prima visita
            retornos_por_estado[estado] = G

        for estado, G in retornos_por_estado.items():
            if estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                idx = estado - 1
                # Aggiornamento incrementale verso il ritorno G
                V[idx] += alpha * (G - V[idx])

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


np.random.seed(0)
V_mc_ejercicio, historial_mc_ejercicio = monte_carlo_primera_visita_ejercicio(200, alpha=0.1)
print("V stimato:", V_mc_ejercicio)
print("Valori reali:", VALORES_REALES)
print(f"RMSE finale: {historial_mc_ejercicio[-1]:.4f}")

# Output atteso approssimativo (con seed 0):
# V stimato: [0.1927 0.2591 0.3563 0.6763 0.8233]
# Valori reali: [0.1667 0.3333 0.5    0.6667 0.8333]
# RMSE finale: 0.0735
```
</details>

### Ora confronta: MC contro TD(0) con diversi tassi di apprendimento

Con le due funzioni già completate, calcola l'RMSE finale —mediato su più esecuzioni, per attenuare il rumore di una singola prova— di MC e di TD(0) per α = 0.01, α = 0.05 e α = 0.15.

**Suggerimenti:**
- Usa `rmse_promedio` per ripetere l'esperimento con più semi e prendere la media dello storico dell'RMSE
- Chiama `monte_carlo_primera_visita_ejercicio` e `td0_prediccion_ejercicio` con ciascun α, su 300 episodi
- L'ultimo valore dello storico mediato è l'RMSE finale da riportare nella tabella

In [ ]:
def rmse_promedio(funcion_prediccion, n_episodios, alpha, n_ejecuciones=20):
    '''
    Repite el experimento n_ejecuciones veces con semillas distintas y
    promedia el historial de RMSE, para suavizar el ruido de una sola
    ejecución antes de comparar tasas de aprendizaje.
    '''
    acumulado = np.zeros(n_episodios)
    for ejecucion in range(n_ejecuciones):
        np.random.seed(ejecucion)
        _, historial = funcion_prediccion(n_episodios, alpha)
        acumulado += np.array(historial)
    return acumulado / n_ejecuciones


ALPHAS_A_PROBAR = [0.01, 0.05, 0.15]
N_EPISODIOS_COMPARACION = 300

print(f"{'α':>6} | {'RMSE MC':>10} | {'RMSE TD(0)':>10}")
for alpha in ALPHAS_A_PROBAR:
    # TODO: usa rmse_promedio con monte_carlo_primera_visita_ejercicio y
    #       quédate con el último valor del historial promediado
    rmse_mc_final = None

    # TODO: haz lo mismo para TD(0) con td0_prediccion_ejercicio
    rmse_td_final = None

    print(f"{alpha:>6} | {rmse_mc_final:>10.4f} | {rmse_td_final:>10.4f}")


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def rmse_promedio(funcion_prediccion, n_episodios, alpha, n_ejecuciones=20):
    """
    Ripete l'esperimento n_ejecuciones volte con semi diversi e calcola la
    media dello storico di RMSE, per attenuare il rumore di una singola
    esecuzione prima di confrontare i tassi di apprendimento.
    """
    acumulado = np.zeros(n_episodios)
    for ejecucion in range(n_ejecuciones):
        np.random.seed(ejecucion)
        _, historial = funcion_prediccion(n_episodios, alpha)
        acumulado += np.array(historial)
    return acumulado / n_ejecuciones


ALPHAS_A_PROBAR = [0.01, 0.05, 0.15]
N_EPISODIOS_COMPARACION = 300

print(f"{'α':>6} | {'RMSE MC':>10} | {'RMSE TD(0)':>10}")
for alpha in ALPHAS_A_PROBAR:
    rmse_mc_final = rmse_promedio(
        monte_carlo_primera_visita_ejercicio, N_EPISODIOS_COMPARACION, alpha
    )[-1]
    rmse_td_final = rmse_promedio(
        td0_prediccion_ejercicio, N_EPISODIOS_COMPARACION, alpha
    )[-1]
    print(f"{alpha:>6} | {rmse_mc_final:>10.4f} | {rmse_td_final:>10.4f}")

# Output atteso approssimativo:
#      α |    RMSE MC | RMSE TD(0)
#   0.01 |     0.0427 |     0.0373
#   0.05 |     0.0571 |     0.0348
#   0.15 |     0.1039 |     0.0640
#
# Osserva lo schema: per ogni α, TD(0) sta sotto MC (varianza minore), e per
# entrambi i metodi l'errore finale cresce con α (più rumore residuo con un
# tasso di apprendimento più alto). È il compromesso tra bias e varianza
# visto in classe, ora misurato con i numeri.
```
</details>

---

## Per continuare a esercitarti

- **Aggiungi un grafico**: usa `matplotlib` per disegnare, sullo stesso asse, l'RMSE in funzione degli episodi per MC e TD(0) con la tua α preferita, e verifica visivamente quello che hai appena visto nella tabella
- **Prova la visita ogni volta**: modifica la tua `monte_carlo_primera_visita_ejercicio` in modo che aggiorni V a ogni occorrenza dello stato nell'episodio, non solo alla prima, e verifica che il risultato finale cambi appena

**Prossima tappa:** nella Lezione 6 riutilizzerai questa stessa idea di differenze temporali per passare dal valutare una politica al migliorarla, con due algoritmi di controllo —SARSA e Q-learning— sull'ambiente di Cliff Walking. Salva questo notebook: l'intuizione di "correggere con ciò che hai appena visto" torna a comparire!